<a href="https://colab.research.google.com/github/rudalshan0412-code/Intent_Classifier-RAG_Chatbot/blob/main/08)_%EC%B5%9C%EC%A2%85_%EB%8B%B5%EB%B3%80_%EC%83%9D%EC%84%B1(LLM).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# LLM은 Large Language Model 의 약자로 자연스럽게 언어를 이해하고 생성하는 모델을 의미함
# RAG 챗봇에서는 검색한 기존 문서에 있던 단순한 문장들을 정리해서 답변으로 만들어주는 역할을 한다

In [ ]:
# 드라이브 연결

from google.colab import drive

drive.mount("/content/drive")

%cd /content/drive/MyDrive/rag_intent_chatbot

Mounted at /content/drive
/content/drive/MyDrive/rag_intent_chatbot


In [ ]:
# 필수 파일 존재 여부 확인

from pathlib import Path

required_files = [
    "models/intent_classifier.pt",
    "data/intents.json",
    "data/documents/sample.txt",
    "src/intent/predict.py",
    "src/rag/document_loader.py",
    "src/rag/text_preprocessor.py",
    "src/rag/chunker.py",
    "src/rag/embedder.py",
    "src/rag/vector_store.py",
    "src/rag/retriever.py",
    "src/rag/answer_generator.py",
    "src/chatbot.py",
    "main.py",
]

all_files_exist = True

for file_path in required_files:
    if Path(file_path).exists():
        print(f"[있음] {file_path}")
    else:
        print(f"[없음] {file_path}")
        all_files_exist = False

print()
print("전체 확인 결과:", "정상" if all_files_exist else "누락 파일 있음")

[있음] models/intent_classifier.pt
[있음] data/intents.json
[있음] data/documents/sample.txt
[있음] src/intent/predict.py
[있음] src/rag/document_loader.py
[있음] src/rag/text_preprocessor.py
[있음] src/rag/chunker.py
[있음] src/rag/embedder.py
[있음] src/rag/vector_store.py
[있음] src/rag/retriever.py
[있음] src/rag/answer_generator.py
[있음] src/chatbot.py
[있음] main.py

전체 확인 결과: 정상


In [ ]:
# SDK 설치(gemini 설치)

!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 19.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.


In [ ]:
# 버전 확인
import importlib.metadata

print(
    "google-genai version:",
    importlib.metadata.version("google-genai"),
)
from google import genai

client = genai.Client(api_key="dummy")
print(hasattr(client, "interactions"))
print(type(client.interactions))

# 사용 모델은 model="gemini-3.5-flash-lite" 입니다.

google-genai version: 2.18.0
True
<class 'google.genai._gaos.google_genai.GeminiNextGenInteractions'>


In [ ]:
# import 확인

from google import genai

print("google-genai import 성공")

google-genai import 성공


In [ ]:
# API 키 준비
from google.colab import userdata

try:
    gemini_api_key = userdata.get("GEMINI_API_KEY")
except Exception as error:
    raise RuntimeError(
        "Colab Secrets에서 GEMINI_API_KEY를 읽지 못했습니다. "
        "왼쪽 Secrets 메뉴에서 키를 등록하고 Notebook access를 활성화해주세요."
    ) from error

if not gemini_api_key:
    raise RuntimeError(
        "GEMINI_API_KEY가 비어 있습니다. "
        "Colab Secrets에 Gemini API 키를 등록해주세요."
    )

print("GEMINI_API_KEY 확인 완료")
print("보안을 위해 실제 키 값은 출력하지 않습니다.")

GEMINI_API_KEY 확인 완료
보안을 위해 실제 키 값은 출력하지 않습니다.


In [ ]:
# 기존 파일 백업

from pathlib import Path
import shutil

backup_pairs = [
    (
        "src/rag/answer_generator.py",
        "src/rag/answer_generator_rule_based_backup.py",
    ),
    (
        "src/chatbot.py",
        "src/chatbot_before_llm_backup.py",
    ),
    (
        "main.py",
        "main_before_llm_backup.py",
    ),
]

for source, backup in backup_pairs:
    source_path = Path(source)
    backup_path = Path(backup)

    if source_path.exists():
        shutil.copy2(source_path, backup_path)
        print(f"[백업 완료] {source} -> {backup}")
    else:
        print(f"[백업 실패] 원본 없음: {source}")

[백업 완료] src/rag/answer_generator.py -> src/rag/answer_generator_rule_based_backup.py
[백업 완료] src/chatbot.py -> src/chatbot_before_llm_backup.py
[백업 완료] main.py -> main_before_llm_backup.py


In [ ]:
# answer_generator 덮어쓰기
# 기존 answer_generator에서 마지막 답변 생성 부분을 LLM 생성으로 변경

In [ ]:
%%writefile src/rag/answer_generator.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any

from src.rag.vector_store import SearchResult


MIN_SEARCH_SCORE = 0.30
MAX_CONTEXT_CHUNKS = 3
MAX_CONTEXT_LENGTH = 1500
ANSWER_PREVIEW_LENGTH = 500

DEFAULT_MODEL_NAME = "gemini-2.5-flash-lite"
DEFAULT_TEMPERATURE = 0.2
DEFAULT_MAX_OUTPUT_TOKENS = 500


@dataclass
class GeneratedAnswer:
    """문서 기반 답변 생성 결과를 저장한다."""

    answer: str
    context: str
    used_results: list[SearchResult]
    sources: list[str]
    has_relevant_context: bool


class AnswerGenerator:
    """검색 결과로 Context를 만들고 LLM 문서 답변을 생성한다."""

    def __init__(
        self,
        client: Any | None,
        model_name: str = DEFAULT_MODEL_NAME,
        min_search_score: float = MIN_SEARCH_SCORE,
        max_context_chunks: int = MAX_CONTEXT_CHUNKS,
        max_context_length: int = MAX_CONTEXT_LENGTH,
        answer_preview_length: int = ANSWER_PREVIEW_LENGTH,
        temperature: float = DEFAULT_TEMPERATURE,
        max_output_tokens: int = DEFAULT_MAX_OUTPUT_TOKENS,
    ) -> None:
        if not 0.0 <= min_search_score <= 1.0:
            raise ValueError(
                "min_search_score는 0 이상 1 이하여야 합니다."
            )

        if max_context_chunks < 1:
            raise ValueError(
                "max_context_chunks는 1 이상이어야 합니다."
            )

        if max_context_length < 1:
            raise ValueError(
                "max_context_length는 1 이상이어야 합니다."
            )

        if answer_preview_length < 1:
            raise ValueError(
                "answer_preview_length는 1 이상이어야 합니다."
            )

        if not 0.0 <= temperature <= 1.0:
            raise ValueError(
                "temperature는 0 이상 1 이하여야 합니다."
            )

        if max_output_tokens < 1:
            raise ValueError(
                "max_output_tokens는 1 이상이어야 합니다."
            )

        if not model_name.strip():
            raise ValueError("model_name이 비어 있습니다.")

        self.client = client
        self.model_name = model_name
        self.min_search_score = min_search_score
        self.max_context_chunks = max_context_chunks
        self.max_context_length = max_context_length
        self.answer_preview_length = answer_preview_length
        self.temperature = temperature
        self.max_output_tokens = max_output_tokens

    def generate(
        self,
        question: str,
        search_results: list[SearchResult],
    ) -> GeneratedAnswer:
        """검색 결과를 정리하고 문서 기반 LLM 답변을 생성한다."""

        if not isinstance(question, str):
            raise TypeError("question은 문자열이어야 합니다.")

        question = question.strip()

        if not question:
            raise ValueError("question이 비어 있습니다.")

        if not search_results:
            return GeneratedAnswer(
                answer=(
                    "관련 문서 내용을 찾지 못했습니다.\n"
                    "질문을 조금 더 구체적으로 입력해주세요."
                ),
                context="",
                used_results=[],
                sources=[],
                has_relevant_context=False,
            )

        score_filtered_results = [
            result
            for result in search_results
            if result.score >= self.min_search_score
        ]

        if not score_filtered_results:
            return GeneratedAnswer(
                answer=(
                    "검색된 내용이 질문과 충분히 관련 있다고 "
                    "판단하기 어렵습니다.\n"
                    "다른 표현으로 질문해주세요."
                ),
                context="",
                used_results=[],
                sources=[],
                has_relevant_context=False,
            )

        usable_results = self._remove_duplicates(
            score_filtered_results
        )

        if not usable_results:
            return GeneratedAnswer(
                answer=(
                    "검색 결과는 존재하지만 사용할 수 있는 "
                    "문서 내용이 없습니다."
                ),
                context="",
                used_results=[],
                sources=[],
                has_relevant_context=False,
            )

        context, used_results = self._build_context(
            usable_results
        )

        if not context or not used_results:
            return GeneratedAnswer(
                answer=(
                    "검색 결과는 존재하지만 사용할 수 있는 "
                    "문서 내용이 없습니다."
                ),
                context="",
                used_results=[],
                sources=[],
                has_relevant_context=False,
            )

        sources = [
            self._format_source(result)
            for result in used_results
        ]

        llm_answer = self._generate_llm_answer(
            question=question,
            context=context,
        )

        answer_lines = [
            "문서 기반 답변:",
            llm_answer,
            "",
            "사용한 출처:",
        ]

        answer_lines.extend(
            f"- {source}"
            for source in sources
        )

        return GeneratedAnswer(
            answer="\n".join(answer_lines),
            context=context,
            used_results=used_results,
            sources=sources,
            has_relevant_context=True,
        )

    def _remove_duplicates(
        self,
        search_results: list[SearchResult],
    ) -> list[SearchResult]:
        """빈 텍스트와 단순 중복 Chunk를 제거한다."""

        unique_results: list[SearchResult] = []
        seen_chunk_keys: set[tuple[str, str]] = set()
        seen_texts: set[str] = set()

        for result in search_results:
            chunk_text = result.chunk.text.strip()

            if not chunk_text:
                continue

            chunk_key = (
                result.chunk.source,
                result.chunk.chunk_id,
            )

            normalized_text = " ".join(
                chunk_text.split()
            )

            if chunk_key in seen_chunk_keys:
                continue

            if normalized_text in seen_texts:
                continue

            seen_chunk_keys.add(chunk_key)
            seen_texts.add(normalized_text)
            unique_results.append(result)

        return unique_results

    def _build_context(
        self,
        search_results: list[SearchResult],
    ) -> tuple[str, list[SearchResult]]:
        """Chunk 개수와 전체 길이를 제한하여 Context를 만든다."""

        context_blocks: list[str] = []
        used_results: list[SearchResult] = []
        current_length = 0

        for result in search_results:
            if len(used_results) >= self.max_context_chunks:
                break

            chunk_text = result.chunk.text.strip()

            if not chunk_text:
                continue

            header = (
                f"[문서 {len(used_results) + 1}]\n"
                f"source: {result.chunk.source}\n"
                f"chunk_id: {result.chunk.chunk_id}\n"
                "내용:\n"
            )

            separator_length = 2 if context_blocks else 0
            remaining_length = (
                self.max_context_length
                - current_length
                - len(header)
                - separator_length
            )

            if remaining_length <= 0:
                break

            limited_text = chunk_text[:remaining_length].strip()

            if not limited_text:
                continue

            context_block = header + limited_text

            context_blocks.append(context_block)
            used_results.append(result)

            current_length += len(context_block)
            current_length += separator_length

        context = "\n\n".join(context_blocks)

        return context, used_results

    def _format_source(
      self,
      result: SearchResult,
  ) -> str:
      """사용한 검색 결과의 출처 정보를 문자열로 만든다."""

      return (
          f"{result.chunk.source}, "
          f"chunk_id={result.chunk.chunk_id}, "
          f"score={result.score:.4f}"
      )

    def _build_user_prompt(
        self,
        question: str,
        context: str,
    ) -> str:
        """질문과 검색 Context를 사용자 프롬프트로 구성한다."""

        return (
            "[질문]\n"
            f"{question}\n\n"
            "[Context]\n"
            f"{context}\n\n"
            "[답변 요청]\n"
            "위 Context에서 질문의 답을 찾으세요. "
            "Context만으로 답할 수 없다면 "
            "'제공된 문서에서 해당 질문의 답을 찾지 못했습니다.'라고 "
            "답하세요."
        )

    def _generate_llm_answer(
        self,
        question: str,
        context: str,
    ) -> str:
        """Gemini API를 호출하고 생성된 텍스트를 반환한다."""

        if self.client is None:
            raise RuntimeError(
                "Gemini 클라이언트가 설정되지 않았습니다. "
                "GEMINI_API_KEY를 확인해주세요."
            )

        system_instruction = (
            "당신은 제공된 문서 Context만 사용해 답하는 "
            "문서 질의응답 도우미입니다.\n"
            "다음 규칙을 반드시 지키세요.\n"
            "1. Context에 있는 정보만 사용하세요.\n"
            "2. Context에 없는 사실을 만들지 마세요.\n"
            "3. 일반 지식으로 내용을 보충하지 마세요.\n"
            "4. 답을 찾을 수 없으면 제공된 문서에서 답을 "
            "찾지 못했다고 명확히 말하세요.\n"
            "5. 답변은 한국어로 작성하세요.\n"
            "6. 불필요하게 장황하게 작성하지 마세요.\n"
            "7. 파일명, Chunk ID, 점수 또는 출처 목록을 "
            "답변에 작성하지 마세요.\n"
            "8. 출처는 Python 프로그램이 별도로 추가합니다."
        )

        user_prompt = self._build_user_prompt(
            question=question,
            context=context,
        )

        try:
            interaction = self.client.interactions.create(
                model=self.model_name,
                system_instruction=system_instruction,
                input=user_prompt,
                generation_config={
                    "temperature": self.temperature,
                    "max_output_tokens": self.max_output_tokens,
                },
            )

            output_text = getattr(
                interaction,
                "output_text",
                None,
            )

            if not output_text:
                raise RuntimeError(
                    "Gemini 응답 객체에서 생성된 텍스트를 "
                    "찾을 수 없습니다."
                )

            answer = str(output_text).strip()

            if not answer:
                raise RuntimeError(
                    "Gemini가 빈 답변을 반환했습니다."
                )

            if len(answer) > self.answer_preview_length:
                answer = (
                    answer[:self.answer_preview_length].rstrip()
                    + "..."
                )

            return answer

        except RuntimeError:
            raise

        except Exception as error:
            self._raise_llm_error(error)

    def _raise_llm_error(
        self,
        error: Exception,
    ) -> None:
        """API 오류를 학습자가 이해하기 쉬운 메시지로 변환한다."""

        error_name = type(error).__name__
        error_message = str(error)
        normalized_message = (
            f"{error_name} {error_message}"
        ).lower()

        if (
            "api key" in normalized_message
            or "unauthenticated" in normalized_message
            or "authentication" in normalized_message
            or "401" in normalized_message
            or "403" in normalized_message
        ):
            message = (
                "Gemini API 인증에 실패했습니다. "
                "GEMINI_API_KEY가 올바른지 확인해주세요."
            )

        elif (
            "429" in normalized_message
            or "rate limit" in normalized_message
            or "resource exhausted" in normalized_message
            or "quota" in normalized_message
        ):
            message = (
                "Gemini API 호출 한도에 도달했습니다. "
                "잠시 후 다시 시도하거나 사용량 및 할당량을 확인해주세요."
            )

        elif (
            "404" in normalized_message
            or "model not found" in normalized_message
            or (
                "model" in normalized_message
                and "not found" in normalized_message
            )
        ):
            message = (
                f"Gemini 모델을 찾을 수 없습니다: "
                f"{self.model_name}. "
                "공식 모델 목록에서 모델명을 확인해주세요."
            )

        elif (
            "token" in normalized_message
            and (
                "limit" in normalized_message
                or "too long" in normalized_message
                or "exceed" in normalized_message
            )
        ):
            message = (
                "질문과 Context가 모델의 입력 길이 제한을 "
                "초과했습니다. MAX_CONTEXT_LENGTH를 줄여주세요."
            )

        elif (
            "connection" in normalized_message
            or "timeout" in normalized_message
            or "503" in normalized_message
            or "unavailable" in normalized_message
        ):
            message = (
                "Gemini API 서버 또는 인터넷 연결 문제로 "
                "호출에 실패했습니다."
            )

        elif (
            "400" in normalized_message
            or "invalid argument" in normalized_message
        ):
            message = (
                "Gemini API 요청 형식이 올바르지 않습니다. "
                "모델명과 generation_config를 확인해주세요."
            )

        else:
            message = (
                "Gemini API 호출 중 예상하지 못한 오류가 "
                f"발생했습니다: {error_name}: {error_message}"
            )

        raise RuntimeError(message) from error

Overwriting src/rag/answer_generator.py


In [ ]:
# 코드 확인

!sed -n '1,420p' src/rag/answer_generator.py

from __future__ import annotations

from dataclasses import dataclass
from typing import Any

from src.rag.vector_store import SearchResult


MIN_SEARCH_SCORE = 0.30
MAX_CONTEXT_CHUNKS = 3
MAX_CONTEXT_LENGTH = 1500
ANSWER_PREVIEW_LENGTH = 500

DEFAULT_MODEL_NAME = "gemini-2.5-flash-lite"
DEFAULT_TEMPERATURE = 0.2
DEFAULT_MAX_OUTPUT_TOKENS = 500


@dataclass
class GeneratedAnswer:
    """문서 기반 답변 생성 결과를 저장한다."""

    answer: str
    context: str
    used_results: list[SearchResult]
    sources: list[str]
    has_relevant_context: bool


class AnswerGenerator:
    """검색 결과로 Context를 만들고 LLM 문서 답변을 생성한다."""

    def __init__(
        self,
        client: Any | None,
        model_name: str = DEFAULT_MODEL_NAME,
        min_search_score: float = MIN_SEARCH_SCORE,
        max_context_chunks: int = MAX_CONTEXT_CHUNKS,
        max_context_length: int = MAX_CONTEXT_LENGTH,
        answer_preview_length: int = ANSWER_PREVIEW_LENGTH,
        temperature: float = DEFAULT_TEMPERATURE,
        max

In [ ]:
# API 키 누락 테스트

from src.rag.answer_generator import AnswerGenerator
from src.rag.chunker import Chunk
from src.rag.vector_store import SearchResult

test_chunk = Chunk(
    text="임베딩은 텍스트의 의미를 숫자 벡터로 표현한 것이다.",
    chunk_id="test_chunk_0000",
    metadata={"source": "test.txt"},
    source="test.txt",
    start_index=0,
    end_index=30,
)

test_result = SearchResult(
    chunk=test_chunk,
    score=0.90,
    rank=1,
)

missing_client_generator = AnswerGenerator(
    client=None,
)

try:
    missing_client_generator.generate(
        question="임베딩은 무엇인가요?",
        search_results=[test_result],
    )
except RuntimeError as error:
    print("[예상된 오류]")
    print(error)

[예상된 오류]
Gemini 클라이언트가 설정되지 않았습니다. GEMINI_API_KEY를 확인해주세요.


In [ ]:
# 가짜 SearchResult 단위 테스트

# 테스트용 클라이언트
class FakeInteraction:
    def __init__(self, output_text: str) -> None:
        self.output_text = output_text


class FakeInteractions:
    def __init__(self) -> None:
        self.call_count = 0
        self.last_arguments = None

    def create(self, **kwargs):
        self.call_count += 1
        self.last_arguments = kwargs

        return FakeInteraction(
            "임베딩은 텍스트의 의미를 숫자 벡터로 표현한 것입니다."
        )


class FakeClient:
    def __init__(self) -> None:
        self.interactions = FakeInteractions()


fake_client = FakeClient()

test_generator = AnswerGenerator(
    client=fake_client,
    min_search_score=0.30,
    max_context_chunks=3,
    max_context_length=1500,
    answer_preview_length=500,
)

In [ ]:
# 검색 결과 없음
class FakeInteraction:
    def __init__(self, output_text: str) -> None:
        self.output_text = output_text


class FakeInteractions:
    def __init__(self) -> None:
        self.call_count = 0
        self.last_arguments = None

    def create(self, **kwargs):
        self.call_count += 1
        self.last_arguments = kwargs

        return FakeInteraction(
            "임베딩은 텍스트의 의미를 숫자 벡터로 표현한 것입니다."
        )


class FakeClient:
    def __init__(self) -> None:
        self.interactions = FakeInteractions()


fake_client = FakeClient()

test_generator = AnswerGenerator(
    client=fake_client,
    min_search_score=0.30,
    max_context_chunks=3,
    max_context_length=1500,
    answer_preview_length=500,
)

In [ ]:
# 낮은 점수

low_score_result = SearchResult(
    chunk=test_chunk,
    score=0.10,
    rank=1,
)

result = test_generator.generate(
    question="임베딩은 무엇인가요?",
    search_results=[low_score_result],
)

print("answer:")
print(result.answer)
print()
print("has_relevant_context:", result.has_relevant_context)
print("used_results 개수:", len(result.used_results))
print("LLM 호출 횟수:", fake_client.interactions.call_count)

assert result.has_relevant_context is False
assert fake_client.interactions.call_count == 0

answer:
검색된 내용이 질문과 충분히 관련 있다고 판단하기 어렵습니다.
다른 표현으로 질문해주세요.

has_relevant_context: False
used_results 개수: 0
LLM 호출 횟수: 0


In [ ]:
# 중복 Chunk 제거

chunk_1 = Chunk(
    text="임베딩은 텍스트 의미를 숫자 벡터로 표현한다.",
    chunk_id="sample_chunk_0001",
    metadata={"source": "sample.txt"},
    source="sample.txt",
    start_index=0,
    end_index=30,
)

chunk_same_id = Chunk(
    text="같은 ID를 가진 중복 Chunk이다.",
    chunk_id="sample_chunk_0001",
    metadata={"source": "sample.txt"},
    source="sample.txt",
    start_index=20,
    end_index=50,
)

chunk_same_text = Chunk(
    text="임베딩은 텍스트 의미를 숫자 벡터로 표현한다.",
    chunk_id="sample_chunk_0002",
    metadata={"source": "sample.txt"},
    source="sample.txt",
    start_index=30,
    end_index=60,
)

chunk_2 = Chunk(
    text="VectorStore는 Chunk와 임베딩을 저장하고 검색한다.",
    chunk_id="sample_chunk_0003",
    metadata={"source": "sample.txt"},
    source="sample.txt",
    start_index=60,
    end_index=100,
)

duplicate_results = [
    SearchResult(chunk=chunk_1, score=0.91, rank=1),
    SearchResult(chunk=chunk_1, score=0.90, rank=2),
    SearchResult(chunk=chunk_same_id, score=0.89, rank=3),
    SearchResult(chunk=chunk_same_text, score=0.88, rank=4),
    SearchResult(chunk=chunk_2, score=0.87, rank=5),
]

before_call_count = fake_client.interactions.call_count

result = test_generator.generate(
    question="임베딩과 VectorStore를 설명해주세요.",
    search_results=duplicate_results,
)

print("answer:")
print(result.answer)
print()
print("context:")
print(result.context)
print()
print("has_relevant_context:", result.has_relevant_context)
print("used_results 개수:", len(result.used_results))
print("sources:", result.sources)
print(
    "이번 테스트의 LLM 호출 횟수:",
    fake_client.interactions.call_count - before_call_count,
)

assert len(result.used_results) == 2
assert result.has_relevant_context is True

answer:
문서 기반 답변:
임베딩은 텍스트의 의미를 숫자 벡터로 표현한 것입니다.

사용한 출처:
- sample.txt, chunk_id=sample_chunk_0001, score=0.9100
- sample.txt, chunk_id=sample_chunk_0003, score=0.8700

context:
[문서 1]
source: sample.txt
chunk_id: sample_chunk_0001
내용:
임베딩은 텍스트 의미를 숫자 벡터로 표현한다.

[문서 2]
source: sample.txt
chunk_id: sample_chunk_0003
내용:
VectorStore는 Chunk와 임베딩을 저장하고 검색한다.

has_relevant_context: True
used_results 개수: 2
sources: ['sample.txt, chunk_id=sample_chunk_0001, score=0.9100', 'sample.txt, chunk_id=sample_chunk_0003, score=0.8700']
이번 테스트의 LLM 호출 횟수: 1


In [ ]:
# 실제 전달 프롬포트 확인

arguments = fake_client.interactions.last_arguments

print("model:")
print(arguments["model"])

print()
print("system_instruction:")
print(arguments["system_instruction"])

print()
print("input:")
print(arguments["input"])

print()
print("generation_config:")
print(arguments["generation_config"])

model:
gemini-2.5-flash-lite

system_instruction:
당신은 제공된 문서 Context만 사용해 답하는 문서 질의응답 도우미입니다.
다음 규칙을 반드시 지키세요.
1. Context에 있는 정보만 사용하세요.
2. Context에 없는 사실을 만들지 마세요.
3. 일반 지식으로 내용을 보충하지 마세요.
4. 답을 찾을 수 없으면 제공된 문서에서 답을 찾지 못했다고 명확히 말하세요.
5. 답변은 한국어로 작성하세요.
6. 불필요하게 장황하게 작성하지 마세요.
7. 파일명, Chunk ID, 점수 또는 출처 목록을 답변에 작성하지 마세요.
8. 출처는 Python 프로그램이 별도로 추가합니다.

input:
[질문]
임베딩과 VectorStore를 설명해주세요.

[Context]
[문서 1]
source: sample.txt
chunk_id: sample_chunk_0001
내용:
임베딩은 텍스트 의미를 숫자 벡터로 표현한다.

[문서 2]
source: sample.txt
chunk_id: sample_chunk_0003
내용:
VectorStore는 Chunk와 임베딩을 저장하고 검색한다.

[답변 요청]
위 Context에서 질문의 답을 찾으세요. Context만으로 답할 수 없다면 '제공된 문서에서 해당 질문의 답을 찾지 못했습니다.'라고 답하세요.

generation_config:
{'temperature': 0.2, 'max_output_tokens': 500}


In [ ]:
# 실제 Gemini 클라이언트 생성

from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise RuntimeError(
        "Colab Secrets에 GEMINI_API_KEY를 등록해주세요."
    )

gemini_client = genai.Client(
    api_key=api_key,
)

print("Gemini 클라이언트 생성 완료")

Gemini 클라이언트 생성 완료


In [ ]:
# LLM 단독 호출 테스트

interaction = gemini_client.interactions.create(
    model="gemini-3.5-flash-lite",
    system_instruction=(
        "답변은 한국어로 짧고 명확하게 작성하세요."
    ),
    input="RAG에서 Retriever의 역할을 한 문장으로 설명하세요.",
    generation_config={
        "temperature": 0.2,
        "max_output_tokens": 150,
    },
)

if not interaction.output_text:
    raise RuntimeError(
        "Gemini 응답에서 output_text를 찾을 수 없습니다."
    )

print(interaction.output_text)

Retriever는 사용자의 질문과 관련된 가장 관련성 높은 문서를 대규모 데이터베이스에서 찾아내는 역할을 합니다.


In [ ]:
# 통합 파이프라인 테스트
llm_answer_generator = AnswerGenerator(
    client=gemini_client,
    model_name="gemini-3.5-flash-lite",
    min_search_score=0.30,
    max_context_chunks=3,
    max_context_length=1500,
    answer_preview_length=500,
    temperature=0.2,
    max_output_tokens=500,
)

fake_llm_results = [
    SearchResult(
        chunk=Chunk(
            text=(
                "임베딩은 텍스트의 의미를 숫자 벡터로 표현한 것이다. "
                "의미가 유사한 문장은 벡터 공간에서도 가까운 위치에 놓인다."
            ),
            chunk_id="embedding_chunk_0000",
            metadata={"source": "embedding_notes.txt"},
            source="embedding_notes.txt",
            start_index=0,
            end_index=70,
        ),
        score=0.92,
        rank=1,
    ),
    SearchResult(
        chunk=Chunk(
            text=(
                "VectorStore는 문서 Chunk와 임베딩을 저장하며, "
                "질문 벡터와 유사한 문서 Chunk를 검색한다."
            ),
            chunk_id="vector_store_chunk_0000",
            metadata={"source": "vector_store_notes.txt"},
            source="vector_store_notes.txt",
            start_index=0,
            end_index=60,
        ),
        score=0.86,
        rank=2,
    ),
]

generated_answer = llm_answer_generator.generate(
    question="임베딩과 VectorStore의 관계를 설명해주세요.",
    search_results=fake_llm_results,
)

print("answer:")
print(generated_answer.answer)

print()
print("context:")
print(generated_answer.context)

print()
print("has_relevant_context:")
print(generated_answer.has_relevant_context)

print()
print("used_results 개수:")
print(len(generated_answer.used_results))

print()
print("sources:")
for source in generated_answer.sources:
    print("-", source)

answer:
문서 기반 답변:
제공된 문서에 따르면, 임베딩은 텍스트의 의미를 숫자 벡터로 표현한 것이고, VectorStore는 문서 Chunk와 그 임베딩을 저장하며 질문 벡터와 유사한 문서 Chunk를 검색하는 역할을 합니다.

사용한 출처:
- embedding_notes.txt, chunk_id=embedding_chunk_0000, score=0.9200
- vector_store_notes.txt, chunk_id=vector_store_chunk_0000, score=0.8600

context:
[문서 1]
source: embedding_notes.txt
chunk_id: embedding_chunk_0000
내용:
임베딩은 텍스트의 의미를 숫자 벡터로 표현한 것이다. 의미가 유사한 문장은 벡터 공간에서도 가까운 위치에 놓인다.

[문서 2]
source: vector_store_notes.txt
chunk_id: vector_store_chunk_0000
내용:
VectorStore는 문서 Chunk와 임베딩을 저장하며, 질문 벡터와 유사한 문서 Chunk를 검색한다.

has_relevant_context:
True

used_results 개수:
2

sources:
- embedding_notes.txt, chunk_id=embedding_chunk_0000, score=0.9200
- vector_store_notes.txt, chunk_id=vector_store_chunk_0000, score=0.8600


In [ ]:
# Retriever 기반 테스트

from main import build_retriever
from src.rag.answer_generator import AnswerGenerator
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise RuntimeError(
        "GEMINI_API_KEY가 등록되지 않았습니다."
    )

client = genai.Client(
    api_key=api_key,
)

retriever = build_retriever()

answer_generator = AnswerGenerator(
    client=client,
    model_name="gemini-3.5-flash-lite",
    min_search_score=0.30,
    max_context_chunks=3,
    max_context_length=1500,
    answer_preview_length=500,
    temperature=0.2,
    max_output_tokens=500,
)

[1/6] 문서를 불러오는 중...
      불러온 문서 수: 1
[2/6] 문서를 전처리하는 중...
[3/6] 문서를 Chunk로 나누는 중...
      생성된 Chunk 수: 9
[4/6] TextEmbedder를 불러오는 중...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/content/drive/MyDrive/rag_intent_chatbot/src/rag/embedder.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = self.model.get_sentence_embedding_dimension() # 출력하는 벡터(임베딩)의 차원 수(크기)를 반환


[5/6] Chunk 임베딩을 생성하는 중...
[6/6] VectorStore와 Retriever를 생성하는 중...
      Retriever 생성 완료


In [ ]:
from main import build_retriever
from src.rag.answer_generator import AnswerGenerator
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise RuntimeError(
        "GEMINI_API_KEY가 등록되지 않았습니다."
    )

client = genai.Client(
    api_key=api_key,
)

retriever = build_retriever()

answer_generator = AnswerGenerator(
    client=client,
    model_name="gemini-3.5-flash-lite",
    min_search_score=0.30,
    max_context_chunks=3,
    max_context_length=1500,
    answer_preview_length=500,
    temperature=0.2,
    max_output_tokens=500,
)
# 전혀 관계 없는 질문도 Chunk를 반환할 수 있기 때문에 검색 점수상으로는 응담이 있으나 문서에 답이 없는 경우를 구분해야함

[1/6] 문서를 불러오는 중...
      불러온 문서 수: 1
[2/6] 문서를 전처리하는 중...
[3/6] 문서를 Chunk로 나누는 중...
      생성된 Chunk 수: 9
[4/6] TextEmbedder를 불러오는 중...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/content/drive/MyDrive/rag_intent_chatbot/src/rag/embedder.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = self.model.get_sentence_embedding_dimension() # 출력하는 벡터(임베딩)의 차원 수(크기)를 반환


[5/6] Chunk 임베딩을 생성하는 중...
[6/6] VectorStore와 Retriever를 생성하는 중...
      Retriever 생성 완료


In [ ]:
# 전체 Chatbot 회귀 테스트

from main import build_chatbot

chatbot = build_chatbot()

chatbot_test_inputs = [
    "안녕하세요",
    "고마워요",
    "너는 누구야?",
    "사용 방법을 알려줘",
    "문서에서 임베딩이 무엇인지 찾아줘",
    "문서에서 Chunk를 사용하는 이유를 찾아줘",
    "자동차 엔진 오일 교환법을 문서에서 찾아줘",
    "파란 생각이 조용하게 숫자를 걸어간다",
]

for user_input in chatbot_test_inputs:
    print("=" * 80)

    try:
        result = chatbot.process_message(
            user_input=user_input,
        )

        print("사용자 입력:", result.user_input)
        print("predicted_intent:", result.predicted_intent)
        print("confidence:", f"{result.confidence:.4f}")
        print("fallback 여부:", result.is_fallback)
        print("requires_rag:", result.requires_rag)
        print("검색 결과 개수:", len(result.search_results))
        print()
        print("최종 response:")
        print(result.response)

    except RuntimeError as error:
        print("사용자 입력:", user_input)
        print("[처리 오류]", error)

    print()

IntentPredictor를 불러오는 중...
[1/6] 문서를 불러오는 중...
      불러온 문서 수: 1
[2/6] 문서를 전처리하는 중...
[3/6] 문서를 Chunk로 나누는 중...
      생성된 Chunk 수: 9
[4/6] TextEmbedder를 불러오는 중...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/content/drive/MyDrive/rag_intent_chatbot/src/rag/embedder.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = self.model.get_sentence_embedding_dimension() # 출력하는 벡터(임베딩)의 차원 수(크기)를 반환


[5/6] Chunk 임베딩을 생성하는 중...
[6/6] VectorStore와 Retriever를 생성하는 중...
      Retriever 생성 완료
Gemini API 키를 확인하는 중...
Gemini 클라이언트를 생성하는 중...
사용자 입력: 안녕하세요
predicted_intent: greeting
confidence: 0.9207
fallback 여부: False
requires_rag: False
검색 결과 개수: 0

최종 response:
안녕하세요. 무엇을 도와드릴까요?

사용자 입력: 고마워요
predicted_intent: fallback
confidence: 0.2976
fallback 여부: True
requires_rag: False
검색 결과 개수: 0

최종 response:
질문의 의도를 확실하게 판단하지 못했습니다. 조금 더 구체적으로 질문해주세요.

사용자 입력: 너는 누구야?
predicted_intent: bot_info
confidence: 0.9749
fallback 여부: False
requires_rag: False
검색 결과 개수: 0

최종 response:
저는 PyTorch 인텐트 분류기와 RAG 검색 기능을 결합한 학습용 챗봇입니다.

사용자 입력: 사용 방법을 알려줘
predicted_intent: help
confidence: 0.9074
fallback 여부: False
requires_rag: False
검색 결과 개수: 0

최종 response:
문서 내용을 묻고 싶다면 '문서에서 해당 내용을 찾아줘'와 같이 질문해주세요.

사용자 입력: 문서에서 임베딩이 무엇인지 찾아줘
predicted_intent: document_query
confidence: 0.9933
fallback 여부: False
requires_rag: True
검색 결과 개수: 3

최종 response:
문서 기반 답변:
제공된 문서에서 해당 질문의 답을 찾지 못했습니다.

사용한 출처:
- sample.txt, 

In [ ]:
# 문서에 없는 질문 테스트

question = "자동차 엔진 오일 교환 주기는 얼마인가요?"

search_results = retriever.retrieve(
    query=question,
    top_k=3,
)

generated_answer = answer_generator.generate(
    question=question,
    search_results=search_results,
)

print("검색 결과:")
for item in search_results:
    print(
        item.rank,
        f"{item.score:.4f}",
        item.chunk.source,
        item.chunk.chunk_id,
    )

print()
print("생성된 Context:")
print(generated_answer.context or "(없음)")

print()
print("최종 답변:")
print(generated_answer.answer)

검색 결과:
1 0.0658 sample.txt sample_chunk_0004
2 0.0288 sample.txt sample_chunk_0005
3 0.0173 sample.txt sample_chunk_0006

생성된 Context:
(없음)

최종 답변:
검색된 내용이 질문과 충분히 관련 있다고 판단하기 어렵습니다.
다른 표현으로 질문해주세요.
